In [51]:
import json
from pathlib import Path
from imgnet.collections.store import IndexedDatasets
from imgnet.collections.source import CompositeSource, HuggingFaceSource, HttpSource
from imgtools.nifti.crawl import Crawler
from rich import print
import pandas as pd

# CancerVerse


In [52]:
DATASET_NAME = "CancerVerse"

indexed_datasets = IndexedDatasets("indexed_datasets") # This will download the latest version of the indexed_datasets if you dont provide a path
print(indexed_datasets.collections[:10])



16:51:38 [info     ] Indexed datasets path: /home/gpudual/bhklab/declan/med-image_index/indexed_datasets [imgnet] call=store.__init__:65


[
    '4D-Lung',
    'ACRIN-6698',
    'ACRIN-Contralateral-Breast-MR',
    'ACRIN-FLT-Breast',
    'ACRIN-NSCLC-FDG-PET',
    'AMOS',
    'Abdomen Atlas 3',
    'Adrenal-ACC-Ki67-Seg',
    'Advanced-MRI-Breast-Lesions',
    'Anti-PD-1_Lung'
]

In [53]:
description = """
We present CancerVerse, a fully open longitudinal and multimodal CT dataset for multicancer screening. It contains 24,422 CT scans from 14,599 patients, followed over time for up to 14.4 years. CT scans were collected at hospitals in Switzerland and Turkey from 2010 to 2025. Human experts drew voxel-wise tumor annotations for 13 cancer types in the pelvis, abdomen, and chest. CT scans are paired with radiology reports, pathology results and clinical variables. CancerVerse also includes CT scans of verified healthy patients for measuring false positives.
"""

In [54]:
source = CompositeSource(
    file_type="nifti",
    sources=[
        HttpSource(
            file_type="nifti",
            url="http://www.cs.jhu.edu/~zongwei/dataset/CancerVerse_Label.tar.gz"
        ),
        HuggingFaceSource(
            file_type="nifti",
            repo_id="BodyMaps/CancerVerse",
            description=description,
            post_download=["unzip"]
        ),
    
    ]

)


In [55]:
print(source)

CompositeSource(
    file_type=<FileType.NIFTI: 'nifti'>,
    post_download=['unzip'],
    source='composite',
    sources=[
        HttpSource(
            file_type=<FileType.NIFTI: 'nifti'>,
            post_download=['unzip'],
            source='http',
            url='http://www.cs.jhu.edu/~zongwei/dataset/CancerVerse_Label.tar.gz',
            description=''
        ),
        HuggingFaceSource(
            file_type=<FileType.NIFTI: 'nifti'>,
            post_download=['unzip'],
            source='huggingface',
            repo_id='BodyMaps/CancerVerse',
            description='\nWe present CancerVerse, a fully open longitudinal and multimodal CT dataset for 
multicancer screening. It contains 24,422 CT scans from 14,599 patients, followed over time for up to 14.4 years. 
CT scans were collected at hospitals in Switzerland and Turkey from 2010 to 2025. Human experts drew voxel-wise 
tumor annotations for 13 cancer types in the pelvis, abdomen, and chest. CT scans are paired with radiology 
reports, pathology results and clinical variables. CancerVerse also includes CT scans of verified healthy patients 
for measuring false positives.\n'
        )
    ]
)

In [56]:
dataset_index_path = (indexed_datasets.path / DATASET_NAME)
dataset_index_path.mkdir(parents=True, exist_ok=True)
(dataset_index_path / "parquet").mkdir(exist_ok=True)
with open(dataset_index_path / "source.json", "w") as f:
    json.dump(source.model_dump(mode="json"), f, indent=2)

In [57]:

print(indexed_datasets.source_config(DATASET_NAME))

CompositeSource(
    file_type=<FileType.NIFTI: 'nifti'>,
    post_download=['unzip'],
    source='composite',
    sources=[
        HttpSource(
            file_type=<FileType.NIFTI: 'nifti'>,
            post_download=['unzip'],
            source='http',
            url='http://www.cs.jhu.edu/~zongwei/dataset/CancerVerse_Label.tar.gz',
            description=''
        ),
        HuggingFaceSource(
            file_type=<FileType.NIFTI: 'nifti'>,
            post_download=['unzip'],
            source='huggingface',
            repo_id='BodyMaps/CancerVerse',
            description='\nWe present CancerVerse, a fully open longitudinal and multimodal CT dataset for 
multicancer screening. It contains 24,422 CT scans from 14,599 patients, followed over time for up to 14.4 years. 
CT scans were collected at hospitals in Switzerland and Turkey from 2010 to 2025. Human experts drew voxel-wise 
tumor annotations for 13 cancer types in the pelvis, abdomen, and chest. CT scans are paired with radiology 
reports, pathology results and clinical variables. CancerVerse also includes CT scans of verified healthy patients 
for measuring false positives.\n'
        )
    ]
)

In [58]:
temp_data_path = Path("temp_data") / DATASET_NAME
temp_data_path.mkdir(parents=True, exist_ok=True)
#downloader = indexed_datasets.downloader(DATASET_NAME).download(temp_data_path)

In [59]:
# import tarfile

# def extract_all_tar_gz(folder_path, extract_to=None, remove_after=False):
#     """
#     Extract all .tar.gz archives in a folder.
    
#     Args:
#         folder_path (str): Path to folder containing .tar.gz files
#         extract_to (str, optional): Directory to extract files to. 
#                                     If None, extracts to same folder as each archive.
#         remove_after (bool): If True, removes the archive after successful extraction.
    
#     Returns:
#         list: List of extracted file paths
#     """
#     folder = Path(folder_path)
#     extracted_files = []
    
#     # Get all .tar.gz files
#     tar_gz_files = list(folder.glob("*.tar.gz"))
    
#     if not tar_gz_files:
#         print(f"No .tar.gz files found in {folder_path}")
#         return extracted_files
    
#     print(f"Found {len(tar_gz_files)} .tar.gz archives")
    
#     for archive_path in tar_gz_files:
#         try:
#             # Determine extraction directory
#             if extract_to:
#                 extract_dir = Path(extract_to)
#             else:
#                 # Extract to same directory as archive, using archive name (without .tar.gz)
#                 extract_dir = folder / archive_path.stem.replace('.tar', '')
            
#             # Create extraction directory if it doesn't exist
#             extract_dir.mkdir(parents=True, exist_ok=True)
            
#             print(f"Extracting: {archive_path.name} -> {extract_dir}")
            
#             # Extract the archive
#             with tarfile.open(archive_path, 'r:gz') as tar:
#                 tar.extractall(path=extract_dir)
#                 extracted_files.extend([str(extract_dir / member.name) for member in tar.getmembers()])
            
#             # Remove archive if requested
#             if remove_after:
#                 archive_path.unlink()
#                 print(f"Removed: {archive_path.name}")
                
#         except Exception as e:
#             print(f"Error extracting {archive_path.name}: {e}")
    
#     return extracted_files

# extract_all_tar_gz(temp_data_path, remove_after=False)

In [60]:

# import os
# import shutil
# import argparse


# def copy_structure_with_files(src_dir, dst_dir, target_filename="kidney_lesion.nii.gz", new_filename="ct.nii.gz"):
#     """
#     Copy the directory structure from src_dir to dst_dir.
#     In each directory that contains target_filename, copy it as new_filename.
#     """
#     src_dir = os.path.abspath(src_dir)
#     dst_dir = os.path.abspath(dst_dir)

#     if not os.path.isdir(src_dir):
#         raise ValueError(f"Source directory does not exist: {src_dir}")

#     copied_count = 0

#     for root, dirs, files in os.walk(src_dir):
#         # Compute relative path from source root
#         rel_path = os.path.relpath(root, src_dir)
#         # Destination directory mirroring the source structure
#         dst_root = os.path.join(dst_dir, rel_path) if rel_path != "." else dst_dir

#         # Create the directory in the destination
#         os.makedirs(dst_root, exist_ok=True)

#         # If the target file exists in this directory, copy + rename it
#         if target_filename in files:
#             src_file = os.path.join(root, target_filename)
#             dst_file = os.path.join(dst_root, new_filename)
#             shutil.copy2(src_file, dst_file)
#             copied_count += 1
#             print(f"Copied: {src_file}  ->  {dst_file}")

#     print(f"\nDone. Copied {copied_count} file(s).")


# copy_structure_with_files("/home/gpudual/bhklab/declan/med-image_index/temp_data/CancerVerse/CancerVerse_Label/CancerVerse/CancerVerse", "/home/gpudual/bhklab/declan/med-image_index/temp_data/CancerVerse/CancerVerse")

# # crawler = Crawler(
# #     nifti_dir=temp_data_path,
# #     scan_name_pattern="image_only/AbdomenAtlas3_images_{drop}/BDMAP_{scan_number}/ct.nii.gz",
# #     mask_name_pattern="mask_only/AbdomenAtlas3_masks_{drop}/BDMAP_{scan_number}/segmentations/{ROINames}.nii.gz",
# #     output_dir=indexed_datasets.path,
# #     n_jobs=10,
# #     force=True,
# #     deep=False,
# # )
# # crawler.crawl()







In [61]:

crawler = Crawler(
    nifti_dir=temp_data_path,
    scan_name_pattern="CancerVerse/{SampleID}/segmentations/ct.nii.gz",
    mask_name_pattern="CancerVerse_Label/CancerVerse/CancerVerse/{SampleID}/segmentations/{ROINames}.nii.gz",
    output_dir=indexed_datasets.path,
    n_jobs=10,
    force=True,
    deep=False,
)
crawler.crawl()

In [62]:
# print(len(index_df))
# index_df

index_df = pd.read_csv(indexed_datasets.path / DATASET_NAME / "index.csv")



def update_filepath(row):
    path = row['filepath']
    if row['file_type'] == 'scan':
        
        return path.replace("/segmentations", "")
    else: 
        return path


index_df['filepath'] = index_df.apply(update_filepath, axis=1)

In [63]:
def map_modality(row):
    if row["file_type"] == "scan":
            return "CT"
    else:
        return "SEG"
    
index_df["Modality"] = index_df.apply(map_modality, axis=1)
index_df["BodyPartExamined"] = "ABDOMEN"




In [64]:
meta_df = pd.read_csv(temp_data_path / "CancerVerse_dataset_metadata.csv")

In [65]:
index_df

,SampleID,ROINames,filepath,file_type,reference_id,reference_scan,Modality,BodyPartExamined
0,CV_00000001,NaN,CancerVerse/CV_00000001/ct.nii.gz,scan,CV_00000001,NaN,CT,ABDOMEN
1,CV_00000002,NaN,CancerVerse/CV_00000002/ct.nii.gz,scan,CV_00000002,NaN,CT,ABDOMEN
2,CV_00000003,NaN,CancerVerse/CV_00000003/ct.nii.gz,scan,CV_00000003,NaN,CT,ABDOMEN
3,CV_00000004,NaN,CancerVerse/CV_00000004/ct.nii.gz,scan,CV_00000004,NaN,CT,ABDOMEN
4,CV_00000005,NaN,CancerVerse/CV_00000005/ct.nii.gz,scan,CV_00000005,NaN,CT,ABDOMEN
...,...,...,...,...,...,...,...,...
341875,CV_00024422,pancreatic_lesion,CancerVerse_Label/CancerVerse/CancerVerse/CV_0...,mask,CV_00024422,CancerVerse/CV_00024422/segmentations/ct.nii.gz,SEG,ABDOMEN
341876,CV_00024422,prostate_lesion,CancerVerse_Label/CancerVerse/CancerVerse/CV_0...,mask,CV_00024422,CancerVerse/CV_00024422/segmentations/ct.nii.gz,SEG,ABDOMEN
341877,CV_00024422,spleen_lesion,CancerVerse_Label/CancerVerse/CancerVerse/CV_0...,mask,CV_00024422,CancerVerse/CV_00024422/segmentations/ct.nii.gz,SEG,ABDOMEN
341878,CV_00024422,stomach_lesion,CancerVerse_Label/CancerVerse/CancerVerse/CV_0...,mask,CV_00024422,CancerVerse/CV_00024422/segmentations/ct.nii.gz,SEG,ABDOMEN


In [66]:
meta_df['scan_number'] = meta_df['CancerVerse ID'].str.extract(r'(\d+)').astype(str)
index_df['scan_number'] = index_df['SampleID']


In [67]:


print(index_df["scan_number"].dtype)

print(index_df.dtypes)

print(meta_df.dtypes)

full_df = index_df.merge(meta_df, left_on="SampleID", how="left", right_on='CancerVerse ID')
full_df

object

SampleID            object
ROINames            object
filepath            object
file_type           object
reference_id        object
reference_scan      object
Modality            object
BodyPartExamined    object
scan_number         object
dtype: object

CancerVerse ID                object
Encrypted Accession Number    object
Patient ID                    object
sex                           object
age                           object
phase                         object
exam_date                     object
scanner                       object
study_description             object
series_description            object
protocol_name                 object
icd10_code                    object
report                        object
scan_number                   object
dtype: object

,SampleID,ROINames,filepath,file_type,reference_id,reference_scan,Modality,BodyPartExamined,scan_number_x,CancerVerse ID,...,age,phase,exam_date,scanner,study_description,series_description,protocol_name,icd10_code,report,scan_number_y
0,CV_00000001,NaN,CancerVerse/CV_00000001/ct.nii.gz,scan,CV_00000001,NaN,CT,ABDOMEN,CV_00000001,CV_00000001,...,85,portal_venous,07-30-2022,NaN,NaN,NaN,NaN,C22.0,- \tPartial regression of the 4 known LI-RADS ...,00000001
1,CV_00000002,NaN,CancerVerse/CV_00000002/ct.nii.gz,scan,CV_00000002,NaN,CT,ABDOMEN,CV_00000002,CV_00000002,...,77,portal_venous,03-28-2019,NaN,NaN,NaN,NaN,C22.0,"In 12/2018, there is a slight size progression...",00000002
2,CV_00000003,NaN,CancerVerse/CV_00000003/ct.nii.gz,scan,CV_00000003,NaN,CT,ABDOMEN,CV_00000003,CV_00000003,...,78,venous,12-19-2023,Philips,Lower Abdomen CT,Parenchyma,Thorax Abdomen Single-phase / Thorax,C16.9,"Thorax, Upper Abdomen, and Pelvis CT \nClinic...",00000003
3,CV_00000004,NaN,CancerVerse/CV_00000004/ct.nii.gz,scan,CV_00000004,NaN,CT,ABDOMEN,CV_00000004,CV_00000004,...,78,portal_venous,07-24-2024,NaN,NaN,NaN,NaN,C78.6,For status post adenocarcinoma of the pancreat...,00000004
4,CV_00000005,NaN,CancerVerse/CV_00000005/ct.nii.gz,scan,CV_00000005,NaN,CT,ABDOMEN,CV_00000005,CV_00000005,...,47,arterial_late,06-25-2022,NaN,NaN,NaN,NaN,NaN,For metastatic breast cancer with bone and liv...,00000005
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
341875,CV_00024422,pancreatic_lesion,CancerVerse_Label/CancerVerse/CancerVerse/CV_0...,mask,CV_00024422,CancerVerse/CV_00024422/segmentations/ct.nii.gz,SEG,ABDOMEN,CV_00024422,CV_00024422,...,NaN,NaN,NaN,Siemens,CT Abdomen and Pelvis - IV Contrast,NaN,CT Abdomen and Pelvis - IV Contrast,NaN,"CLINICAL INFORMATION:\nAbdominal pain, gallsto...",00024422
341876,CV_00024422,prostate_lesion,CancerVerse_Label/CancerVerse/CancerVerse/CV_0...,mask,CV_00024422,CancerVerse/CV_00024422/segmentations/ct.nii.gz,SEG,ABDOMEN,CV_00024422,CV_00024422,...,NaN,NaN,NaN,Siemens,CT Abdomen and Pelvis - IV Contrast,NaN,CT Abdomen and Pelvis - IV Contrast,NaN,"CLINICAL INFORMATION:\nAbdominal pain, gallsto...",00024422
341877,CV_00024422,spleen_lesion,CancerVerse_Label/CancerVerse/CancerVerse/CV_0...,mask,CV_00024422,CancerVerse/CV_00024422/segmentations/ct.nii.gz,SEG,ABDOMEN,CV_00024422,CV_00024422,...,NaN,NaN,NaN,Siemens,CT Abdomen and Pelvis - IV Contrast,NaN,CT Abdomen and Pelvis - IV Contrast,NaN,"CLINICAL INFORMATION:\nAbdominal pain, gallsto...",00024422
341878,CV_00024422,stomach_lesion,CancerVerse_Label/CancerVerse/CancerVerse/CV_0...,mask,CV_00024422,CancerVerse/CV_00024422/segmentations/ct.nii.gz,SEG,ABDOMEN,CV_00024422,CV_00024422,...,NaN,NaN,NaN,Siemens,CT Abdomen and Pelvis - IV Contrast,NaN,CT Abdomen and Pelvis - IV Contrast,NaN,"CLINICAL INFORMATION:\nAbdominal pain, gallsto...",00024422


In [68]:
from imgnet.collections.store import IndexedDatasets
from imgindex.model import validate_index
from imgindex.parquet import csv_to_parquet

index_df = full_df
#index_df.rename(columns={"reference_id": "SampleID"}, inplace=True)
index_df["SampleID"] = index_df["SampleID"].astype(str)
index_df.to_csv(indexed_datasets.path / DATASET_NAME / "index.csv", index=False)
csv_to_parquet(index_df, indexed_datasets.path / DATASET_NAME)
validate_index(index_df, "nifti", lazy=True)

,SampleID,ROINames,filepath,file_type,reference_id,reference_scan,Modality,BodyPartExamined,scan_number_x,CancerVerse ID,...,age,phase,exam_date,scanner,study_description,series_description,protocol_name,icd10_code,report,scan_number_y
0,CV_00000001,NaN,CancerVerse/CV_00000001/ct.nii.gz,scan,CV_00000001,NaN,CT,ABDOMEN,CV_00000001,CV_00000001,...,85,portal_venous,07-30-2022,NaN,NaN,NaN,NaN,C22.0,- \tPartial regression of the 4 known LI-RADS ...,00000001
1,CV_00000002,NaN,CancerVerse/CV_00000002/ct.nii.gz,scan,CV_00000002,NaN,CT,ABDOMEN,CV_00000002,CV_00000002,...,77,portal_venous,03-28-2019,NaN,NaN,NaN,NaN,C22.0,"In 12/2018, there is a slight size progression...",00000002
2,CV_00000003,NaN,CancerVerse/CV_00000003/ct.nii.gz,scan,CV_00000003,NaN,CT,ABDOMEN,CV_00000003,CV_00000003,...,78,venous,12-19-2023,Philips,Lower Abdomen CT,Parenchyma,Thorax Abdomen Single-phase / Thorax,C16.9,"Thorax, Upper Abdomen, and Pelvis CT \nClinic...",00000003
3,CV_00000004,NaN,CancerVerse/CV_00000004/ct.nii.gz,scan,CV_00000004,NaN,CT,ABDOMEN,CV_00000004,CV_00000004,...,78,portal_venous,07-24-2024,NaN,NaN,NaN,NaN,C78.6,For status post adenocarcinoma of the pancreat...,00000004
4,CV_00000005,NaN,CancerVerse/CV_00000005/ct.nii.gz,scan,CV_00000005,NaN,CT,ABDOMEN,CV_00000005,CV_00000005,...,47,arterial_late,06-25-2022,NaN,NaN,NaN,NaN,NaN,For metastatic breast cancer with bone and liv...,00000005
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
341875,CV_00024422,pancreatic_lesion,CancerVerse_Label/CancerVerse/CancerVerse/CV_0...,mask,CV_00024422,CancerVerse/CV_00024422/segmentations/ct.nii.gz,SEG,ABDOMEN,CV_00024422,CV_00024422,...,NaN,NaN,NaN,Siemens,CT Abdomen and Pelvis - IV Contrast,NaN,CT Abdomen and Pelvis - IV Contrast,NaN,"CLINICAL INFORMATION:\nAbdominal pain, gallsto...",00024422
341876,CV_00024422,prostate_lesion,CancerVerse_Label/CancerVerse/CancerVerse/CV_0...,mask,CV_00024422,CancerVerse/CV_00024422/segmentations/ct.nii.gz,SEG,ABDOMEN,CV_00024422,CV_00024422,...,NaN,NaN,NaN,Siemens,CT Abdomen and Pelvis - IV Contrast,NaN,CT Abdomen and Pelvis - IV Contrast,NaN,"CLINICAL INFORMATION:\nAbdominal pain, gallsto...",00024422
341877,CV_00024422,spleen_lesion,CancerVerse_Label/CancerVerse/CancerVerse/CV_0...,mask,CV_00024422,CancerVerse/CV_00024422/segmentations/ct.nii.gz,SEG,ABDOMEN,CV_00024422,CV_00024422,...,NaN,NaN,NaN,Siemens,CT Abdomen and Pelvis - IV Contrast,NaN,CT Abdomen and Pelvis - IV Contrast,NaN,"CLINICAL INFORMATION:\nAbdominal pain, gallsto...",00024422
341878,CV_00024422,stomach_lesion,CancerVerse_Label/CancerVerse/CancerVerse/CV_0...,mask,CV_00024422,CancerVerse/CV_00024422/segmentations/ct.nii.gz,SEG,ABDOMEN,CV_00024422,CV_00024422,...,NaN,NaN,NaN,Siemens,CT Abdomen and Pelvis - IV Contrast,NaN,CT Abdomen and Pelvis - IV Contrast,NaN,"CLINICAL INFORMATION:\nAbdominal pain, gallsto...",00024422
